# Entraînement du World Model — Bras de découpe (sessions)

**Runtime recommandé : GPU (T4 ou A100)**

Pipeline :
```
1. Clone du repo GitHub
2. Installation des dépendances
3. Génération des formes de pièces  →  pieces_database.json
4. Simulation en mode session       →  dataset/session_SSS_pieceNNNN.npz
                                        dataset/session_SSS_deviations.npy
                                        dataset/session_SSS_cadence.npy
5. Entraînement (split par session) →  world_model/checkpoints/best_model.pt
6. Visualisation : loss + rollout autorégressif
```

Nouveautés par rapport à l'ancienne version :
- **Mode session** : chaque session = cadence fixe + vitesse fixe sur N pièces, dégradation progressive
- **Historique K=100** : le modèle voit les 100 dernières déviations pour conditionner sa prédiction
- **Split par session** : le train/val est séparé au niveau session (pas épisode)
- **Rollout autorégressif** : évaluation toutes les N epochs sur des sessions complètes de val

## 0. Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Aucun GPU détecté. Aller dans Exécution → Modifier le type d'exécution → GPU.")

## 1. Montage Google Drive (optionnel — uniquement pour sauvegarder les checkpoints)

Les données sont **toujours générées directement sur le serveur Colab**.  
Drive est utilisé uniquement pour persister les checkpoints entre sessions.  
Mettre `USE_DRIVE = False` pour ignorer complètement Drive.

In [ ]:
USE_DRIVE = True  # False = ne pas sauvegarder les checkpoints sur Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/chain_simple_decoupe'
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Drive monté → {DRIVE_DIR} (sauvegarde checkpoints uniquement)")
else:
    DRIVE_DIR = None
    print("Drive non utilisé — télécharge manuellement les checkpoints après l'entraînement.")

## 2. Clone du repo et installation

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/industrial-world-model.git'
REPO_DIR = '/content/industrial-world-model'
WORK_DIR = os.path.join(REPO_DIR, 'arm_decoupe')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(WORK_DIR)
print(f"Répertoire de travail : {os.getcwd()}")

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dépendances installées.")

## 3. Paramètres du run

Ajuste selon le temps disponible.  
Référence : 200 sessions × 100 pièces ≈ 15 min de simulation sur Colab CPU.

In [ ]:
# ── Dataset (mode session) ────────────────────────────────────────────────────
NUM_SHAPES           = 100    # Formes de pièces uniques dans pieces_database.json
N_SESSIONS           = 200    # Nombre de sessions de production à simuler
N_PIECES_PER_SESSION = 100    # Pièces par session (dégradation progressive sur cette plage)
SPEED_MIN            = 1.0    # Durée min par segment (s) — fixe pour toute la session
SPEED_MAX            = 3.0    # Durée max par segment (s)
SEED                 = 42

# ── Entraînement ─────────────────────────────────────────────────────────────
EPOCHS               = 120
BATCH_SIZE           = 32
LR                   = 3e-4
EARLY_STOPPING       = 40
VAL_SPLIT            = 0.1    # Fraction de sessions réservées à la validation

# Poids des loss qualité
LAMBDA_DEV           = 1.0    # MSE sur cut_deviation
LAMBDA_DEFECT        = 0.5    # BCE sur cut_defect

# Rollout autorégressif (évaluation sur sessions complètes)
ROLLOUT_EVAL_EVERY   = 10     # Évaluer toutes les N epochs (0 = jamais)
N_ROLLOUT_SESSIONS   = 5      # Nombre de sessions de val utilisées

# ── Architecture ─────────────────────────────────────────────────────────────
SHAPE_EMBED_DIM      = 256
H_DIM                = 512
GRU_LAYERS           = 3

# ── Chemins ──────────────────────────────────────────────────────────────────
DATASET_DIR  = 'dataset'
DB_PATH      = 'pieces_database.json'
SAVE_DIR     = 'world_model/checkpoints'

TOTAL_EPISODES = N_SESSIONS * N_PIECES_PER_SESSION
print(f"Dataset attendu : {N_SESSIONS} sessions × {N_PIECES_PER_SESSION} pièces = {TOTAL_EPISODES} épisodes")
print(f"Historique modèle : K=100 déviations passées par pièce")

## 4. Génération des formes de pièces

Produit `pieces_database.json` avec les waypoints de découpe.

In [ ]:
!python generate_pieces.py -n {NUM_SHAPES} --seed {SEED}
print("pieces_database.json généré.")

## 5. Simulation et enregistrement du dataset (mode session)

Génère les fichiers `session_SSS_pieceNNNN.npz` + `session_SSS_deviations.npy` + `session_SSS_cadence.npy` **directement sur le serveur Colab**.

La dégradation est modélisée via la loi de Hill sur les frottements et le bruit moteur.  
**Durée estimée : ~15 min pour 200 sessions × 100 pièces sur Colab T4.**

In [ ]:
import os

os.makedirs(DATASET_DIR, exist_ok=True)

print(f"Génération de {TOTAL_EPISODES} épisodes ({N_SESSIONS} sessions × {N_PIECES_PER_SESSION} pièces)...")
!python record_dataset.py \
    --mode session \
    --n-sessions {N_SESSIONS} \
    --n-pieces   {N_PIECES_PER_SESSION} \
    --speed-min  {SPEED_MIN} \
    --speed-max  {SPEED_MAX} \
    --seed       {SEED} \
    --out        {DATASET_DIR}

import glob
final_count = len(glob.glob(os.path.join(DATASET_DIR, 'session_*_piece*.npz')))
print(f"\nTotal épisodes générés : {final_count}")

## 6. Entraînement du World Model

In [ ]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# Le module utilise des imports relatifs → appel via -m depuis le répertoire arm_decoupe
!python -m world_model.train \
    --data_dir             {DATASET_DIR} \
    --db_path              {DB_PATH} \
    --save_dir             {SAVE_DIR} \
    --epochs               {EPOCHS} \
    --batch_size           {BATCH_SIZE} \
    --lr                   {LR} \
    --early_stopping       {EARLY_STOPPING} \
    --val_split            {VAL_SPLIT} \
    --lambda_dev           {LAMBDA_DEV} \
    --lambda_defect        {LAMBDA_DEFECT} \
    --rollout_eval_every   {ROLLOUT_EVAL_EVERY} \
    --n_rollout_sessions   {N_ROLLOUT_SESSIONS} \
    --shape_embed_dim      {SHAPE_EMBED_DIM} \
    --h_dim                {H_DIM} \
    --gru_layers           {GRU_LAYERS} \
    --seed                 {SEED} \
    --use_amp              True \
    --num_workers          2

## 7. Sauvegarde des checkpoints sur Drive

In [ ]:
if USE_DRIVE:
    import shutil
    drive_ckpt_dir = f'{DRIVE_DIR}/checkpoints'
    if os.path.exists(SAVE_DIR):
        shutil.copytree(SAVE_DIR, drive_ckpt_dir, dirs_exist_ok=True)
        print(f"Checkpoints sauvegardés → {drive_ckpt_dir}")
        for f in sorted(os.listdir(drive_ckpt_dir)):
            size = os.path.getsize(os.path.join(drive_ckpt_dir, f)) / 1e6
            print(f"  {f}  ({size:.1f} MB)")
    else:
        print("Aucun checkpoint trouvé.")
else:
    print("Drive non utilisé — télécharge manuellement les fichiers dans", SAVE_DIR)

## 8. Visualisation des courbes d'entraînement

In [ ]:
from IPython.display import Image, display
import os

curves_path = os.path.join(SAVE_DIR, 'training_curves.png')
if os.path.exists(curves_path):
    display(Image(curves_path))
    print("Courbes : loss train/val, lr, grad norm, rollout MSE autorégressif")
else:
    print("Courbes introuvables — l'entraînement a peut-être échoué.")

## 9. Résumé du modèle entraîné

In [ ]:
import torch, os

ckpt_path = os.path.join(SAVE_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print("=== Modèle sauvegardé ===")
    print(f"  Epoch              : {ckpt.get('epoch', '?')}")
    print(f"  Meilleure val loss : {ckpt.get('val_loss', '?'):.6f}")
    print(f"  dev_mean           : {ckpt.get('dev_mean', '?'):.6f} m")
    print(f"  dev_std            : {ckpt.get('dev_std',  '?'):.6f} m")
    H = ckpt.get('hyperparams', {})
    if H:
        print("  Hyperparamètres clés :")
        for k in ['epochs', 'batch_size', 'lr', 'h_dim', 'gru_layers',
                  'lambda_dev', 'lambda_defect', 'val_split']:
            if k in H:
                print(f"    {k}: {H[k]}")
    n_params = sum(v.numel() for v in ckpt['model_state'].values())
    print(f"  Paramètres totaux  : {n_params:,}")
else:
    print("Checkpoint introuvable.")